# Demo: turn the gripper clockwise in the air

A first test before dipping with rotation: the **empty** gripper goes up into the air above `mid_station`,
pointing straight down, and only the last joint turns, so the gripper spins in place.

- **Cell 1**: settings (`ROTATION_SPEED_DEG_S`, `TOTAL_TURN_DEG`), then up into the air.
- **Cell 2**: a short 30-degree turn so you can tell the code which way is clockwise seen from above; then the
  gripper winds back to the counterclockwise end of its range and turns clockwise by `TOTAL_TURN_DEG` at
  `ROTATION_SPEED_DEG_S`. You can change the two settings in cell 1 and run cells 1 and 2 again.
- **Cell 3**: turns the gripper back and brings the arm back to where it started.

The last joint can turn between -360 and +360 degrees, so one clockwise turn can be at most about 2 full turns
(700 degrees here, keeping 10 degrees spare at each end).

**Before running:** gripper empty, nothing attached to it that could wind up, hand on the e-stop. Do not run
anything in another notebook that is connected to the arm at the same time. Run the cells in order.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='rotation_demo', connect_db=False)
arm = rt.arm

In [ ]:
# ===== Demo 1/3: settings, then take the EMPTY gripper up into the air, pointing straight down =====
# Route: straight up from where the arm is -> across, high above the bench, to above mid_station.
# Before running: gripper EMPTY, nothing attached to the gripper that could wind up, hand on the e-stop.
import time

ROTATION_SPEED_DEG_S = 10.0   # how fast the gripper turns clockwise in the demo, degrees per second (at most 180)
TOTAL_TURN_DEG = 360.0        # how far it turns clockwise in the demo, degrees (at most 700, almost 2 turns)
WIND_SPEED_DEG_S = 30.0       # speed for winding back and unwinding, before and after the demo

DEMO_SPOT = [347.9, 308.1, 500.0, 180.0, 0.0, 90.0]   # high above mid_station, gripper pointing straight down
J7_LIMIT = 350.0              # the last joint turns between -360 and +360 degrees; stay 10 degrees inside that

if not 0 < ROTATION_SPEED_DEG_S <= 180 or not 0 < WIND_SPEED_DEG_S <= 180:
    raise SystemExit('The speeds must be between 0 and 180 degrees per second. The arm did not move.')
if not 0 < TOTAL_TURN_DEG <= 2 * J7_LIMIT:
    raise SystemExit(f'TOTAL_TURN_DEG must be between 0 and {2 * J7_LIMIT:g}. The arm did not move.')


def at_speed(mm_s):
    """The arm's default motion settings with the speed swapped to mm_s."""
    return {**arm.config.pos_settings_dict['default'], 'speed': mm_s}


def j7():
    """Angle of the last joint (the one that turns the gripper), in degrees."""
    return arm.get_joint_angles()[6]


def turn_j7_to(angle, speed):
    """Turn only the last joint, to angle (degrees), at speed (degrees per second). Nothing else moves."""
    arm.set_servo_angle(servo_id=7, angle=angle, speed=speed, mvacc=200, wait=True)


start = arm.get_cartesian_pos()          # remembered, so cell 3 can bring the arm back here
print('Arm is now at:', [round(v, 1) for v in start])
pointing_down = abs(((start[3] - 180) + 180) % 360 - 180) <= 5 and abs(start[4]) <= 5
if not pointing_down or start[2] < 240:
    raise SystemExit('The gripper must point straight down and be at least 5 cm above the racks (z >= 240) '
                     'before this demo. The arm did not move.')

steps = [
    ('straight up from here', [start[0], start[1], DEMO_SPOT[2], *start[3:]], 30),
    ('across, high above the bench, to above mid_station', DEMO_SPOT, 30),
]
for name, pose, speed in steps:
    print(f'  {name:52s} -> {[round(v, 1) for v in pose]} at {speed} mm/s')
if input('Gripper empty, nothing attached that could wind up, hand on the e-stop? Type yes to start: '
         ).strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
for name, pose, speed in steps:
    print('->', name)
    arm.move_to_pos(pose, **at_speed(speed))
j7_home = j7()                           # cell 3 turns the gripper back to this angle
print(f'In the air at {[round(v, 1) for v in arm.get_cartesian_pos()]}, last joint at {j7_home:.1f} degrees.')
print('Next: run cell 2.')

In [ ]:
# ===== Demo 2/3: turn the gripper CLOCKWISE (seen from above), in the air, at ROTATION_SPEED_DEG_S =====
# a) a short 30-degree turn, so you can tell the code which way is clockwise seen from above;
# b) wind the gripper back to the counterclockwise end, so the full turn fits inside the joint's range;
# c) turn clockwise by TOTAL_TURN_DEG at ROTATION_SPEED_DEG_S. Only the last joint moves.
here = arm.get_cartesian_pos()
if max(abs(a - b) for a, b in zip(here[:3], DEMO_SPOT[:3])) > 5:
    raise SystemExit('The arm is not up in the air at the demo spot. Run cell 1 first. Nothing moved.')

print(f'a) short turn: 30 degrees at 15 degrees per second. Watch it from ABOVE.')
turn_j7_to(j7_home + 30, 15)
answer = ''
while answer not in ('yes', 'no'):
    answer = input('Seen from above, did the gripper turn CLOCKWISE? Type yes or no: ').strip().lower()
turn_j7_to(j7_home, 15)
cw = 1 if answer == 'yes' else -1         # +1: clockwise = the last joint's angle going up
print('   noted: clockwise seen from above =', 'joint angle going UP' if cw == 1 else 'joint angle going DOWN')

ccw_end = -cw * J7_LIMIT                  # as far counterclockwise as the joint goes
cw_end = ccw_end + cw * TOTAL_TURN_DEG
wind_s = abs(ccw_end - j7_home) / WIND_SPEED_DEG_S
turn_s = TOTAL_TURN_DEG / ROTATION_SPEED_DEG_S
print(f'b) wind back counterclockwise: {abs(ccw_end - j7_home):.0f} degrees at {WIND_SPEED_DEG_S:g} deg/s '
      f'(about {wind_s:.0f} s)')
print(f'c) turn clockwise: {TOTAL_TURN_DEG:g} degrees ({TOTAL_TURN_DEG / 360:.2f} turns) at '
      f'{ROTATION_SPEED_DEG_S:g} deg/s (about {turn_s:.0f} s)')
if input('Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled. The gripper is back where it was.')
arm.assert_ready()
turn_j7_to(ccw_end, WIND_SPEED_DEG_S)
print(f'   wound back; last joint at {j7():.1f} degrees. Now turning clockwise...')
t0 = time.time()
turn_j7_to(cw_end, ROTATION_SPEED_DEG_S)
print(f'   done: turned {abs(j7() - ccw_end):.0f} degrees clockwise in {time.time() - t0:.1f} s; '
      f'last joint at {j7():.1f} degrees.')
print('Run cell 3 to turn it back and bring the arm back.')
# If the arm stops with an error while winding back or turning, the last joint's range may be narrower than
# +-360 degrees on this arm: tell Claude the error, and use a smaller TOTAL_TURN_DEG meanwhile.

In [ ]:
# ===== Demo 3/3: turn the gripper back, then bring the arm back to where it started =====
print(f'Turning the gripper back to {j7_home:.1f} degrees at {WIND_SPEED_DEG_S:g} deg/s...')
arm.assert_ready()
turn_j7_to(j7_home, WIND_SPEED_DEG_S)
print('Going back: across, high above the bench, then down to the start position.')
arm.move_to_pos([start[0], start[1], DEMO_SPOT[2], *start[3:]], **at_speed(30))
arm.move_to_pos(start, **at_speed(20))
print('Back at:', [round(v, 1) for v in arm.get_cartesian_pos()], f'; last joint at {j7():.1f} degrees.')